In [1]:
!pip install pandas numpy matplotlib scikit-learn lifelines pycox torchtuples torch openpyxl

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 13.3 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=49a00fb48ce81457adb613956335f8a3d805b280083a2e9a0d53bba1a81f79f5
  Stored in directory: /root/.cache/p

In [2]:
!pip install scikit-survival

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 53.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 108.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 18.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [3]:
# ==========================================
# CELL 1: Install & Import Libraries (DeepSurv)
# ==========================================

# Uncomment if first time running
# !pip install pandas numpy matplotlib scikit-learn lifelines pycox torchtuples torch openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

import torch
import torchtuples as tt

from pycox.models import CoxPH
from pycox.evaluation import EvalSurv

from lifelines.utils import concordance_index

import warnings
warnings.filterwarnings("ignore")

# ==========================================
# Global configuration
# ==========================================

N_REPEATS = 25
RANDOM_STATE_BASE = 42
TIME_GRID = np.arange(0.25, 5.0, 0.25)

# DeepSurv specific config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Enhanced DeepSurv Pipeline Initialized")
print(f"Using device: {DEVICE}")

Enhanced DeepSurv Pipeline Initialized
Using device: cuda


In [5]:
# ==========================================
# CELL 2: Load & Preprocess Data (DeepSurv)
# ==========================================

file_path = r"/kaggle/input/datasets/sweta2000/umhs-dataset/Training.xlsx"

data = pd.read_excel(file_path)

# ==========================================
# Encode categorical variables
# ==========================================

data["Race"] = np.where(data["Race"] == "Caucasian", 1, 0)
data["Smoking"] = np.where(data["Smoking"] == "Current", 1, 0)
data["Alcohol"] = np.where(data["Alcohol"] == "Yes", 1, 0)
data["Drug"] = np.where(data["Drug"] == "Yes", 1, 0)

# ==========================================
# Convert numeric safely
# ==========================================

data["HGB"] = pd.to_numeric(data["HGB"], errors="coerce")
data["Duration"] = pd.to_numeric(data["Duration"], errors="coerce")
data["EndEvent"] = pd.to_numeric(data["EndEvent"], errors="coerce")
data["MAGGIC"] = pd.to_numeric(data["MAGGIC"], errors="coerce")

# ==========================================
# Create survival columns
# ==========================================

data["time"] = data["Duration"].astype("float32")
data["event"] = data["EndEvent"].astype("int64")

data.drop(columns=["Duration", "EndEvent"], inplace=True)

# Remove duplicate columns
data = data.loc[:, ~data.columns.duplicated()]

print("Data Loaded Successfully")
print("Shape:", data.shape)

Data Loaded Successfully
Shape: (343, 157)


In [6]:
# ==========================================
# CELL 3: Create D, M, DM subsets (DeepSurv)
# ==========================================

# Get column list
cols = list(data.columns)

# Find index positions
idx_race = cols.index("Race")
idx_c_sa = cols.index("C_SA")
idx_rvpower_s = cols.index("RVpowerIndex_S")

# ==========================================
# Digital features subset (D)
# ==========================================

D = pd.concat([
    data.iloc[:, idx_race:idx_c_sa],
    data[["time", "event"]]
], axis=1)

# ==========================================
# MRI features subset (M)
# ==========================================

M = pd.concat([
    data.iloc[:, idx_c_sa:idx_rvpower_s + 1],
    data[["time", "event"]]
], axis=1)

# ==========================================
# Combined features subset (DM)
# ==========================================

DM = pd.concat([
    data.iloc[:, idx_race:idx_rvpower_s + 1],
    data[["time", "event"]]
], axis=1)

# ==========================================
# Remove duplicate columns
# ==========================================

D = D.loc[:, ~D.columns.duplicated()]
M = M.loc[:, ~M.columns.duplicated()]
DM = DM.loc[:, ~DM.columns.duplicated()]

# ==========================================
# Reset index (important for DeepSurv)
# ==========================================

D = D.reset_index(drop=True)
M = M.reset_index(drop=True)
DM = DM.reset_index(drop=True)

# ==========================================
# Print summary
# ==========================================


print("D shape:", D.shape)
print("M shape:", M.shape)
print("DM shape:", DM.shape)

D shape: (343, 69)
M shape: (343, 82)
DM shape: (343, 149)


In [7]:
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
import torchtuples as tt
import torch
from pycox.models import CoxPH
from lifelines.utils import concordance_index


def deepsurv_variable_hunting(dataset, model_name, n_repeats=25):

    print(f"\nStarting DeepSurv Variable Hunting for {model_name}")

    feature_names = list(dataset.columns)
    feature_names.remove("time")
    feature_names.remove("event")

    freq_counter = defaultdict(int)
    importance_counter = defaultdict(float)

    # Prepare data
    X_full = dataset.drop(columns=["time", "event"]).values.astype("float32")
    time_full = dataset["time"].values.astype("float32")
    event_full = dataset["event"].values.astype("int64")

    # Impute
    imputer = KNNImputer(n_neighbors=5)
    X_full = imputer.fit_transform(X_full)

    # Scale
    scaler = StandardScaler()
    X_full = scaler.fit_transform(X_full).astype("float32")

    X_tensor = torch.from_numpy(X_full)

    for rep in range(n_repeats):

        print(f"{model_name} Repeat {rep+1}/{n_repeats}")

        torch.manual_seed(RANDOM_STATE_BASE + rep)

        # Define DeepSurv network
        net = tt.practical.MLPVanilla(
            in_features=X_full.shape[1],
            num_nodes=[32, 16],
            out_features=1,
            batch_norm=True,
            dropout=0.1
        )

        model = CoxPH(net, tt.optim.Adam)

        model.fit(
            X_full,
            (time_full, event_full),
            batch_size=256,
            epochs=100,
            verbose=False
        )

        # Baseline prediction
        baseline_pred = model.predict(X_full).reshape(-1)

        baseline_cindex = concordance_index(
            time_full,
            -baseline_pred,
            event_full
        )

        importance_dict = {}

        # Permutation importance
        for i, feature in enumerate(feature_names):

            X_permuted = X_full.copy()
            np.random.shuffle(X_permuted[:, i])

            perm_pred = model.predict(X_permuted).reshape(-1)

            perm_cindex = concordance_index(
                time_full,
                -perm_pred,
                event_full
            )

            importance = baseline_cindex - perm_cindex

            importance_dict[feature] = importance

        importances = np.array(list(importance_dict.values()))
        threshold = np.mean(importances)

        selected_features = [
            feature for feature, imp in importance_dict.items()
            if imp >= threshold
        ]

        for feature in selected_features:

            freq_counter[feature] += 1
            importance_counter[feature] += importance_dict[feature]

    avg_importance = {
        feature: importance_counter[feature] / n_repeats
        for feature in importance_counter
    }

    sorted_features = sorted(
        avg_importance.items(),
        key=lambda x: x[1],
        reverse=True
    )

    sorted_features_dict = dict(sorted_features)

    print(f"\nCompleted DeepSurv Variable Hunting for {model_name}")

    return {
        "frequency": dict(freq_counter),
        "importance": sorted_features_dict
    }

In [8]:
result_D_ds = deepsurv_variable_hunting(D, "D", N_REPEATS)
result_M_ds = deepsurv_variable_hunting(M, "M", N_REPEATS)
result_DM_ds = deepsurv_variable_hunting(DM, "DM", N_REPEATS)


Starting DeepSurv Variable Hunting for D
D Repeat 1/25
D Repeat 2/25
D Repeat 3/25
D Repeat 4/25
D Repeat 5/25
D Repeat 6/25
D Repeat 7/25
D Repeat 8/25
D Repeat 9/25
D Repeat 10/25
D Repeat 11/25
D Repeat 12/25
D Repeat 13/25
D Repeat 14/25
D Repeat 15/25
D Repeat 16/25
D Repeat 17/25
D Repeat 18/25
D Repeat 19/25
D Repeat 20/25
D Repeat 21/25
D Repeat 22/25
D Repeat 23/25
D Repeat 24/25
D Repeat 25/25

Completed DeepSurv Variable Hunting for D

Starting DeepSurv Variable Hunting for M
M Repeat 1/25
M Repeat 2/25
M Repeat 3/25
M Repeat 4/25
M Repeat 5/25
M Repeat 6/25
M Repeat 7/25
M Repeat 8/25
M Repeat 9/25
M Repeat 10/25
M Repeat 11/25
M Repeat 12/25
M Repeat 13/25
M Repeat 14/25
M Repeat 15/25
M Repeat 16/25
M Repeat 17/25
M Repeat 18/25
M Repeat 19/25
M Repeat 20/25
M Repeat 21/25
M Repeat 22/25
M Repeat 23/25
M Repeat 24/25
M Repeat 25/25

Completed DeepSurv Variable Hunting for M

Starting DeepSurv Variable Hunting for DM
DM Repeat 1/25
DM Repeat 2/25
DM Repeat 3/25
DM Repeat 

In [9]:
print("\nTop features in D (DeepSurv):")
print(list(result_D_ds["importance"].items())[:10])

print("\nTop features in M (DeepSurv):")
print(list(result_M_ds["importance"].items())[:10])

print("\nTop features in DM (DeepSurv):")
print(list(result_DM_ds["importance"].items())[:10])


Top features in D (DeepSurv):
[('HGB', np.float64(0.02293444771045083)), ('NYHA class', np.float64(0.02025285972305841)), ('EAr_D', np.float64(0.01826397988454864)), ('Alcohol', np.float64(0.01819031766830754)), ('DM', np.float64(0.017885752735772215)), ('CO_D', np.float64(0.016857314870559907)), ('HR', np.float64(0.014099231504763252)), ('PH', np.float64(0.01370683854517122)), ('MVr_D', np.float64(0.012722314693487263)), ('minRVP_D', np.float64(0.011902114247264221))]

Top features in M (DeepSurv):
[('Amref_LV', np.float64(0.018186067925062853)), ('C_SA', np.float64(0.01781350710061267)), ('Hed_LW_S', np.float64(0.01588837341077309)), ('LVIDd_S', np.float64(0.01546623224846832)), ('EAr_S', np.float64(0.015055423734816014)), ('LVm_S', np.float64(0.014192725856146197)), ('R_SA', np.float64(0.01347026950455077)), ('LVESV_S', np.float64(0.012940468180047446)), ('AVr_S', np.float64(0.012900803909763786)), ('R_tSA', np.float64(0.012444664801501566))]

Top features in DM (DeepSurv):
[('HGB'

In [11]:
result_D_ds
result_M_ds
result_DM_ds

{'frequency': {'Race': 9,
  'Age': 21,
  'Gender': 17,
  'Alcohol': 22,
  'Drug': 20,
  'HGB': 24,
  'NYHA class': 24,
  'Height': 13,
  'HR': 22,
  'Usage of ERA': 2,
  'O2 Supp': 11,
  'CAD': 9,
  'AF': 18,
  'HLD': 12,
  'DM': 25,
  'COPD': 21,
  'CTD': 11,
  'ICDorPacemaker': 22,
  'Renal failure': 22,
  "Lateral E/e'": 10,
  'SBP_D': 8,
  'RVEDP_D': 13,
  'PCWP_D': 12,
  'LVIDs_D': 9,
  'LVIDd_D': 15,
  'LVEDV_D': 15,
  'RVEDV_D': 10,
  'RVESV_D': 9,
  'LVEDP_D': 15,
  'MVr_D': 11,
  'PVR_D': 16,
  'RVSWI_D': 2,
  'C_SA': 21,
  'C_PA': 8,
  'k_act_LV': 11,
  'Vw_RV': 7,
  'Vw_SEP': 12,
  'Amref_RV': 11,
  'Amref_SEP': 8,
  'K_P': 10,
  'B_P': 11,
  'R_PA': 10,
  'R_tPA': 19,
  'R_p_c': 10,
  'R_m_o': 13,
  'SBP_S': 9,
  'RVSP_S': 3,
  'RVEDP_S': 9,
  'PADP_S': 6,
  'PCWPmax_S': 8,
  'LVIDd_S': 14,
  'CO_S': 7,
  'minLVP_S': 10,
  'AVr_S': 11,
  'MVmg_S': 6,
  'AVpg_S': 8,
  'EAr_S': 20,
  'StressRV_S': 7,
  'LVpower_S': 11,
  'PVR_S': 8,
  'Smoking': 11,
  'Creatinine': 6,
  'HTN'

In [13]:
def extract_feature_groups(result, n_repeats=25):

    freq = result["frequency"]
    importance = result["importance"]

    # Mandatory features: ≥80% frequency
    mandatory = [
        feature for feature, count in freq.items()
        if count >= 0.8 * n_repeats
    ]

    # Valid features: ≥5 repeats
    valid = [
        feature for feature, count in freq.items()
        if count >= 5
    ]

    # Sorted valid features by importance
    sorted_valid = {
        feature: importance[feature]
        for feature in valid
        if feature in importance
    }

    sorted_valid = dict(
        sorted(sorted_valid.items(),
               key=lambda x: x[1],
               reverse=True)
    )

    return mandatory, valid, sorted_valid

In [14]:
mandatory_D_ds, valid_D_ds, sorted_valid_D_ds = extract_feature_groups(result_D_ds)
mandatory_M_ds, valid_M_ds, sorted_valid_M_ds = extract_feature_groups(result_M_ds)
mandatory_DM_ds, valid_DM_ds, sorted_valid_DM_ds = extract_feature_groups(result_DM_ds)


print("\nMandatory D (DeepSurv):", len(mandatory_D_ds))
print("Valid D (DeepSurv):", len(valid_D_ds))

print("\nMandatory M (DeepSurv):", len(mandatory_M_ds))
print("Valid M (DeepSurv):", len(valid_M_ds))

print("\nMandatory DM (DeepSurv):", len(mandatory_DM_ds))
print("Valid DM (DeepSurv):", len(valid_DM_ds))


Mandatory D (DeepSurv): 9
Valid D (DeepSurv): 55

Mandatory M (DeepSurv): 1
Valid M (DeepSurv): 73

Mandatory DM (DeepSurv): 12
Valid DM (DeepSurv): 126


In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
import torchtuples as tt
import torch
from pycox.models import CoxPH
from lifelines.utils import concordance_index


def train_deepsurv_model(dataset, features, random_state):

    # Prepare data
    X = dataset[features].values.astype("float32")
    time = dataset["time"].values.astype("float32")
    event = dataset["event"].values.astype("int64")

    # Impute
    imputer = KNNImputer(n_neighbors=5)
    X = imputer.fit_transform(X)

    # Scale (mandatory for DeepSurv)
    scaler = StandardScaler()
    X = scaler.fit_transform(X).astype("float32")

    # Set seed
    torch.manual_seed(random_state)

    # Define network
    net = tt.practical.MLPVanilla(
        in_features=X.shape[1],
        num_nodes=[32, 16],
        out_features=1,
        batch_norm=True,
        dropout=0.1
    )

    model = CoxPH(net, tt.optim.Adam)

    # Train
    model.fit(
        X,
        (time, event),
        batch_size=256,
        epochs=100,
        verbose=False
    )

    # Predict risk scores
    risk_scores = model.predict(X).reshape(-1)

    # Calculate C-index
    cindex = concordance_index(
        time,
        -risk_scores,
        event
    )

    return cindex

In [16]:
def find_best_features_deepsurv(dataset, sorted_valid_features,
                                mandatory_features, model_name,
                                max_additional=12, repeats=10):

    print(f"\nSelecting best features for DeepSurv {model_name}")

    selected = mandatory_features.copy()
    remaining = list(sorted_valid_features.keys())

    remaining = [f for f in remaining if f not in selected]

    history = []

    scores = []

    for rep in range(repeats):

        cindex = train_deepsurv_model(
            dataset,
            selected,
            RANDOM_STATE_BASE + rep
        )

        scores.append(cindex)

    best_score = np.mean(scores)

    history.append((len(selected), best_score, selected.copy()))

    print(f"Initial features: {len(selected)} | C-index: {best_score:.4f}")

    # Forward selection
    for i in range(max_additional):

        best_feature = None
        best_feature_score = best_score

        for feature in remaining:

            test_features = selected + [feature]

            scores = []

            for rep in range(repeats):

                cindex = train_deepsurv_model(
                    dataset,
                    test_features,
                    RANDOM_STATE_BASE + rep
                )

                scores.append(cindex)

            mean_score = np.mean(scores)

            if mean_score > best_feature_score:

                best_feature_score = mean_score
                best_feature = feature

        if best_feature is None:
            break

        selected.append(best_feature)
        remaining.remove(best_feature)
        best_score = best_feature_score

        history.append((len(selected), best_score, selected.copy()))

        print(f"Added: {best_feature} | Total: {len(selected)} | C-index: {best_score:.4f}")

    return selected, history

In [17]:
selected_D_ds, history_D_ds = find_best_features_deepsurv(
    D, sorted_valid_D_ds, mandatory_D_ds, "D"
)

selected_M_ds, history_M_ds = find_best_features_deepsurv(
    M, sorted_valid_M_ds, mandatory_M_ds, "M"
)

selected_DM_ds, history_DM_ds = find_best_features_deepsurv(
    DM, sorted_valid_DM_ds, mandatory_DM_ds, "DM"
)

print("\nFinal DeepSurv selected features:")
print("D:", len(selected_D_ds))
print("M:", len(selected_M_ds))
print("DM:", len(selected_DM_ds))


Selecting best features for DeepSurv D
Initial features: 9 | C-index: 0.8125
Added: BNP | Total: 10 | C-index: 0.8295
Added: RVESV_D | Total: 11 | C-index: 0.8377
Added: AVr_D | Total: 12 | C-index: 0.8422
Added: minRVP_D | Total: 13 | C-index: 0.8454
Added: Septal E/e' | Total: 14 | C-index: 0.8525
Added: CAD | Total: 15 | C-index: 0.8587
Added: Gender | Total: 16 | C-index: 0.8738
Added: minLVP_D | Total: 17 | C-index: 0.8739
Added: LVm_D | Total: 18 | C-index: 0.8857

Selecting best features for DeepSurv M
Initial features: 1 | C-index: 0.5632
Added: LVESP_S | Total: 2 | C-index: 0.6715
Added: LVIDd_S | Total: 3 | C-index: 0.7192
Added: PVR_S | Total: 4 | C-index: 0.7428
Added: R_SA | Total: 5 | C-index: 0.7576
Added: AVr_S | Total: 6 | C-index: 0.7711
Added: Vw_SEP | Total: 7 | C-index: 0.7809
Added: TVr_S | Total: 8 | C-index: 0.7863
Added: R_m_o | Total: 9 | C-index: 0.7979
Added: Vw_LV | Total: 10 | C-index: 0.8044
Added: k_act_LV | Total: 11 | C-index: 0.8121
Added: LVm_S | To

In [19]:
# ==========================================
# CELL: Train-Test Split (DeepSurv)
# ==========================================

from sklearn.model_selection import train_test_split

def split_dataset_deepsurv(dataset, selected_features, model_name, test_size=0.3):

    print(f"\nSplitting dataset for DeepSurv {model_name}")

    # Features
    X = dataset[selected_features].values.astype("float32")

    # Targets
    time = dataset["time"].values.astype("float32")
    event = dataset["event"].values.astype("int64")

    # Split
    X_train, X_test, time_train, time_test, event_train, event_test = train_test_split(
        X,
        time,
        event,
        test_size=test_size,
        random_state=42
    )

    print(f"{model_name} Train shape:", X_train.shape)
    print(f"{model_name} Test shape:", X_test.shape)

    return X_train, X_test, time_train, time_test, event_train, event_test

In [20]:
# Digital
X_train_D_ds, X_test_D_ds, time_train_D_ds, time_test_D_ds, event_train_D_ds, event_test_D_ds = \
    split_dataset_deepsurv(D, selected_D_ds, "D")

# MRI
X_train_M_ds, X_test_M_ds, time_train_M_ds, time_test_M_ds, event_train_M_ds, event_test_M_ds = \
    split_dataset_deepsurv(M, selected_M_ds, "M")

# Combined
X_train_DM_ds, X_test_DM_ds, time_train_DM_ds, time_test_DM_ds, event_train_DM_ds, event_test_DM_ds = \
    split_dataset_deepsurv(DM, selected_DM_ds, "DM")


Splitting dataset for DeepSurv D
D Train shape: (240, 18)
D Test shape: (103, 18)

Splitting dataset for DeepSurv M
M Train shape: (240, 13)
M Test shape: (103, 13)

Splitting dataset for DeepSurv DM
DM Train shape: (240, 24)
DM Test shape: (103, 24)


In [21]:
# ==========================================
# CELL: Hyperparameter tuning with Cross-Validation (DeepSurv)
# ==========================================

from sklearn.model_selection import KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
import torchtuples as tt
import torch
from pycox.models import CoxPH
from lifelines.utils import concordance_index


def tune_deepsurv_cv(X_train, time_train, event_train, model_name, n_splits=5):

    print(f"\nHyperparameter tuning with CV for DeepSurv {model_name}")

    # Parameter grid optimized for small dataset (343 patients)
    param_grid = {
        "num_nodes": [[16, 8], [32, 16]],
        "dropout": [0.0, 0.1, 0.2],
        "lr": [1e-3, 5e-4],
        "batch_size": [32, 64],
        "epochs": [100, 150]
    }

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    best_score = -1
    best_params = None

    for params in ParameterGrid(param_grid):

        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):

            # Split fold
            X_fold_train = X_train[train_idx]
            X_fold_val   = X_train[val_idx]

            time_fold_train = time_train[train_idx]
            time_fold_val   = time_train[val_idx]

            event_fold_train = event_train[train_idx]
            event_fold_val   = event_train[val_idx]

            # Impute
            imputer = KNNImputer(n_neighbors=5)
            X_fold_train = imputer.fit_transform(X_fold_train)
            X_fold_val   = imputer.transform(X_fold_val)

            # Scale
            scaler = StandardScaler()
            X_fold_train = scaler.fit_transform(X_fold_train).astype("float32")
            X_fold_val   = scaler.transform(X_fold_val).astype("float32")

            # Set seed
            torch.manual_seed(42)

            # Build network
            net = tt.practical.MLPVanilla(
                in_features=X_fold_train.shape[1],
                num_nodes=params["num_nodes"],
                out_features=1,
                batch_norm=True,
                dropout=params["dropout"]
            )

            model = CoxPH(net, tt.optim.Adam)

            model.optimizer.set_lr(params["lr"])

            # Train
            model.fit(
                X_fold_train,
                (time_fold_train, event_fold_train),
                batch_size=params["batch_size"],
                epochs=params["epochs"],
                verbose=False
            )

            # Predict
            risk_scores = model.predict(X_fold_val).reshape(-1)

            # Evaluate
            cindex = concordance_index(
                time_fold_val,
                -risk_scores,
                event_fold_val
            )

            fold_scores.append(cindex)

        mean_score = np.mean(fold_scores)
        std_score  = np.std(fold_scores)

        print(f"{params} → Mean CV C-index: {mean_score:.4f} (±{std_score:.4f})")

        if mean_score > best_score:

            best_score = mean_score
            best_params = params

    print(f"\nBest params for DeepSurv {model_name}:")
    print(best_params)
    print(f"Best CV C-index: {best_score:.4f}")

    return best_params

In [22]:
best_params_D_ds = tune_deepsurv_cv(
    X_train_D_ds,
    time_train_D_ds,
    event_train_D_ds,
    "D"
)

best_params_M_ds = tune_deepsurv_cv(
    X_train_M_ds,
    time_train_M_ds,
    event_train_M_ds,
    "M"
)

best_params_DM_ds = tune_deepsurv_cv(
    X_train_DM_ds,
    time_train_DM_ds,
    event_train_DM_ds,
    "DM"
)


Hyperparameter tuning with CV for DeepSurv D
{'batch_size': 32, 'dropout': 0.0, 'epochs': 100, 'lr': 0.001, 'num_nodes': [16, 8]} → Mean CV C-index: 0.6655 (±0.1070)
{'batch_size': 32, 'dropout': 0.0, 'epochs': 100, 'lr': 0.001, 'num_nodes': [32, 16]} → Mean CV C-index: 0.6858 (±0.0634)
{'batch_size': 32, 'dropout': 0.0, 'epochs': 100, 'lr': 0.0005, 'num_nodes': [16, 8]} → Mean CV C-index: 0.6880 (±0.1119)
{'batch_size': 32, 'dropout': 0.0, 'epochs': 100, 'lr': 0.0005, 'num_nodes': [32, 16]} → Mean CV C-index: 0.6973 (±0.0664)
{'batch_size': 32, 'dropout': 0.0, 'epochs': 150, 'lr': 0.001, 'num_nodes': [16, 8]} → Mean CV C-index: 0.6574 (±0.0864)
{'batch_size': 32, 'dropout': 0.0, 'epochs': 150, 'lr': 0.001, 'num_nodes': [32, 16]} → Mean CV C-index: 0.6802 (±0.0522)
{'batch_size': 32, 'dropout': 0.0, 'epochs': 150, 'lr': 0.0005, 'num_nodes': [16, 8]} → Mean CV C-index: 0.6642 (±0.1145)
{'batch_size': 32, 'dropout': 0.0, 'epochs': 150, 'lr': 0.0005, 'num_nodes': [32, 16]} → Mean CV C-in

In [29]:
from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc, integrated_brier_score

def train_and_evaluate_test_deepsurv(
        X_train, time_train, event_train,
        X_test, time_test, event_test,
        best_params, model_name):

    print(f"\nFinal DeepSurv training and testing for {model_name}")

    # Impute
    imputer = KNNImputer(n_neighbors=5)
    X_train = imputer.fit_transform(X_train)
    X_test  = imputer.transform(X_test)

    # Scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype("float32")
    X_test  = scaler.transform(X_test).astype("float32")

    # Train model
    torch.manual_seed(42)

    net = tt.practical.MLPVanilla(
        in_features=X_train.shape[1],
        num_nodes=best_params["num_nodes"],
        out_features=1,
        batch_norm=True,
        dropout=best_params["dropout"]
    )

    model = CoxPH(net, tt.optim.Adam)
    model.optimizer.set_lr(best_params["lr"])

    model.fit(
        X_train,
        (time_train, event_train),
        batch_size=best_params["batch_size"],
        epochs=best_params["epochs"],
        verbose=False
    )

    # Risk scores
    risk_scores_test = model.predict(X_test).reshape(-1)

    # sksurv format
    y_train_sksurv = Surv.from_arrays(event_train.astype(bool), time_train)
    y_test_sksurv  = Surv.from_arrays(event_test.astype(bool), time_test)

    # C-index
    cindex = concordance_index(
        time_test,
        -risk_scores_test,
        event_test
    )

    # Time grid
    max_time = min(time_train.max(), time_test.max())

    valid_times = np.linspace(
        time_test.min() + 1e-6,
        max_time * 0.99,
        20
    )

    # iAUC
    auc_times, auc_scores = cumulative_dynamic_auc(
        y_train_sksurv,
        y_test_sksurv,
        risk_scores_test,
        valid_times
    )

    iauc = np.mean(auc_scores)

    # Survival functions
    model.compute_baseline_hazards()
    surv_funcs = model.predict_surv_df(X_test)

    # Interpolate survival probabilities
    surv_matrix = np.zeros((X_test.shape[0], len(valid_times)))

    for i in range(X_test.shape[0]):
        surv_matrix[i, :] = np.interp(
            valid_times,
            surv_funcs.index.values,
            surv_funcs.iloc[:, i].values
        )

    # Brier score
    brier = integrated_brier_score(
        y_train_sksurv,
        y_test_sksurv,
        surv_matrix,
        valid_times
    )

    print(f"\n{model_name} TEST Results:")
    print(f"C-index: {cindex:.4f}")
    print(f"iAUC:    {iauc:.4f}")
    print(f"Brier:   {brier:.4f}")

    return model, scaler, imputer, cindex, iauc, brier

In [30]:
# ==========================================
# CELL: Run DeepSurv final training and evaluation on TEST set
# ==========================================

model_D_ds, scaler_D_ds, imputer_D_ds, cindex_D_ds, iauc_D_ds, brier_D_ds = \
train_and_evaluate_test_deepsurv(
    X_train_D_ds, time_train_D_ds, event_train_D_ds,
    X_test_D_ds, time_test_D_ds, event_test_D_ds,
    best_params_D_ds,
    "DeepSurv D"
)


model_M_ds, scaler_M_ds, imputer_M_ds, cindex_M_ds, iauc_M_ds, brier_M_ds = \
train_and_evaluate_test_deepsurv(
    X_train_M_ds, time_train_M_ds, event_train_M_ds,
    X_test_M_ds, time_test_M_ds, event_test_M_ds,
    best_params_M_ds,
    "DeepSurSurv M"
)


model_DM_ds, scaler_DM_ds, imputer_DM_ds, cindex_DM_ds, iauc_DM_ds, brier_DM_ds = \
train_and_evaluate_test_deepsurv(
    X_train_DM_ds, time_train_DM_ds, event_train_DM_ds,
    X_test_DM_ds, time_test_DM_ds, event_test_DM_ds,
    best_params_DM_ds,
    "DeepSurv DM"
)


Final DeepSurv training and testing for DeepSurv D

DeepSurv D TEST Results:
C-index: 0.7488
iAUC:    0.7710
Brier:   0.1448

Final DeepSurv training and testing for DeepSurSurv M

DeepSurSurv M TEST Results:
C-index: 0.6455
iAUC:    0.6788
Brier:   0.2088

Final DeepSurv training and testing for DeepSurv DM

DeepSurv DM TEST Results:
C-index: 0.6874
iAUC:    0.6813
Brier:   0.1977


In [31]:
test_data = pd.read_excel("/kaggle/input/datasets/sweta2000/umhs-dataset/Testing.xlsx")

# Apply SAME preprocessing
test_data["Race"] = np.where(test_data["Race"] == "Caucasian", 1, 0)
test_data["Smoking"] = np.where(test_data["Smoking"] == "Current", 1, 0)
test_data["Alcohol"] = np.where(test_data["Alcohol"] == "Yes", 1, 0)
test_data["Drug"] = np.where(test_data["Drug"] == "Yes", 1, 0)

test_data["time"] = test_data["Duration"]
test_data["event"] = test_data["EndEvent"]

test_data.drop(columns=["Duration", "EndEvent"], inplace=True)

In [32]:
# ==========================================
# CELL: Final DeepSurv training and evaluation on EXTERNAL dataset
# ==========================================

from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc, integrated_brier_score
from lifelines.utils import concordance_index

def train_and_evaluate_external_deepsurv(train_dataset, test_dataset,
                                         selected_features, best_params,
                                         model_name):

    print(f"\nFINAL DeepSurv Training and External Evaluation: {model_name}")

    # ==========================================
    # Prepare training data
    # ==========================================
    X_train = train_dataset[selected_features].values.astype("float32")
    time_train = train_dataset["time"].values.astype("float32")
    event_train = train_dataset["event"].values.astype("int64")

    # ==========================================
    # Prepare external test data
    # ==========================================
    X_test = test_dataset[selected_features].values.astype("float32")
    time_test = test_dataset["time"].values.astype("float32")
    event_test = test_dataset["event"].values.astype("int64")

    # ==========================================
    # Impute (fit on train, apply on test)
    # ==========================================
    imputer = KNNImputer(n_neighbors=5)

    X_train = imputer.fit_transform(X_train)
    X_test  = imputer.transform(X_test)

    # ==========================================
    # Scale (fit on train, apply on test)
    # ==========================================
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train).astype("float32")
    X_test  = scaler.transform(X_test).astype("float32")

    # ==========================================
    # Train DeepSurv model on FULL training dataset
    # ==========================================
    torch.manual_seed(42)

    net = tt.practical.MLPVanilla(
        in_features=X_train.shape[1],
        num_nodes=best_params["num_nodes"],
        out_features=1,
        batch_norm=True,
        dropout=best_params["dropout"]
    )

    model = CoxPH(net, tt.optim.Adam)
    model.optimizer.set_lr(best_params["lr"])

    model.fit(
        X_train,
        (time_train, event_train),
        batch_size=best_params["batch_size"],
        epochs=best_params["epochs"],
        verbose=False
    )

    # ==========================================
    # Risk scores
    # ==========================================
    risk_scores_test = model.predict(X_test).reshape(-1)

    # ==========================================
    # sksurv format
    # ==========================================
    y_train_sksurv = Surv.from_arrays(event_train.astype(bool), time_train)
    y_test_sksurv  = Surv.from_arrays(event_test.astype(bool), time_test)

    # ==========================================
    # C-index
    # ==========================================
    cindex = concordance_index(
        time_test,
        -risk_scores_test,
        event_test
    )

    # ==========================================
    # Time grid
    # ==========================================
    max_time = min(time_train.max(), time_test.max())

    valid_times = np.linspace(
        time_test.min() + 1e-6,
        max_time * 0.99,
        20
    )

    # ==========================================
    # iAUC
    # ==========================================
    auc_times, auc_scores = cumulative_dynamic_auc(
        y_train_sksurv,
        y_test_sksurv,
        risk_scores_test,
        valid_times
    )

    iauc = np.mean(auc_scores)

    # ==========================================
    # Survival functions
    # ==========================================
    model.compute_baseline_hazards()

    surv_funcs = model.predict_surv_df(X_test)

    # Interpolate survival probabilities
    surv_matrix = np.zeros((X_test.shape[0], len(valid_times)))

    for i in range(X_test.shape[0]):
        surv_matrix[i, :] = np.interp(
            valid_times,
            surv_funcs.index.values,
            surv_funcs.iloc[:, i].values
        )

    # ==========================================
    # Integrated Brier Score
    # ==========================================
    brier = integrated_brier_score(
        y_train_sksurv,
        y_test_sksurv,
        surv_matrix,
        valid_times
    )

    # ==========================================
    # Print results
    # ==========================================
    print(f"\n{model_name} External Results:")
    print(f"C-index: {cindex:.4f}")
    print(f"iAUC:    {iauc:.4f}")
    print(f"Brier:   {brier:.4f}")

    return model, scaler, imputer, cindex, iauc, brier

In [33]:
# Digital
model_D_ext_ds, scaler_D_ext_ds, imputer_D_ext_ds, cindex_D_ext_ds, iauc_D_ext_ds, brier_D_ext_ds = \
train_and_evaluate_external_deepsurv(
    data, test_data,
    selected_D_ds,
    best_params_D_ds,
    "DeepSurv D"
)

# MRI
model_M_ext_ds, scaler_M_ext_ds, imputer_M_ext_ds, cindex_M_ext_ds, iauc_M_ext_ds, brier_M_ext_ds = \
train_and_evaluate_external_deepsurv(
    data, test_data,
    selected_M_ds,
    best_params_M_ds,
    "DeepSurv M"
)

# Combined
model_DM_ext_ds, scaler_DM_ext_ds, imputer_DM_ext_ds, cindex_DM_ext_ds, iauc_DM_ext_ds, brier_DM_ext_ds = \
train_and_evaluate_external_deepsurv(
    data, test_data,
    selected_DM_ds,
    best_params_DM_ds,
    "DeepSurv DM"
)


FINAL DeepSurv Training and External Evaluation: DeepSurv D

DeepSurv D External Results:
C-index: 0.5844
iAUC:    0.6546
Brier:   0.2557

FINAL DeepSurv Training and External Evaluation: DeepSurv M

DeepSurv M External Results:
C-index: 0.6182
iAUC:    0.6433
Brier:   0.2209

FINAL DeepSurv Training and External Evaluation: DeepSurv DM

DeepSurv DM External Results:
C-index: 0.6376
iAUC:    0.6916
Brier:   0.2347
